# Lab 2 — Predicting a Number, and Measuring It Honestly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/INCORTX/INCORTX.github.io/blob/master/DataAnalytics/session-02/lab_02.ipynb)

**Data Analytics · Lab session 2**

Session 1 answered with a **label** — survived or not. This session the answer is a **number**, and that
changes how you measure everything. Being wrong by 2 and being wrong by 200 are not the same mistake,
and MAE, RMSE and R² each take a different view of that.

### How the session runs

| | Part | What happens |
|---|---|---|
| **1** | 🎤 **Assignment 1 on screen — 60 min** | **The hour opens with presentations of last session's work.** Two speakers per group, four minutes, then two of questions. |
| **2** | 🎬 **Demo — 45 min** | Blocks A to C. The instructor walks it; you watch. Do not type along — you keep this file. |
| **3** | 📋 **Pick a topic** | Your group claims one of the fifteen. First come, first served. |
| **4** | 🟠 **Your hour — 60 min** | The section at the bottom. The same five steps as session 1, on a number. |

---
## 🎤 The 45 minutes we actually walk through

**This notebook's demo half holds about 45 minutes of material and the slot is 45.** The nine below are the ones we walk together — **about 40 minutes.** Everything else is reference you keep. *(This table is a map, not something read aloud.)*

| | Walked in the demo | Why this one earns the time |
|:--:|---|---|
| 1 | **How the session runs** | so the hour is not spent guessing what to hand in |
| 2 | **A.1** — the same Pipeline, and what its three steps are for | impute · scale · model, and **why it is one object and not three lines** |
| 3 | **A.2** — a new baseline | **and why its R² comes out negative**, which every class asks |
| 4 | **A.3** — MAE, RMSE and R² | three numbers, three different questions |
| 5 | **A.4** — choosing between them | **from what an error costs, not from which looks better** |
| 6 | **B.1** — the residual plot | a cloud means random error; a shape means the model is wrong in a pattern |
| 7 | **B.2** — the target that hit a ceiling | **965 districts stuck at 5.00001**, and the residual plot shows it instantly |
| 8 | **B.3** — outliers, IQR and the fence | the formula flags 466 rows; **most of them are real** |
| 9 | **Part 2 — picking your topic** | you skim and claim, not read out loud |

**B.4 and block C are yours to read** — about 6 minutes if you sit down with them.

> **Regression only this session.** Every one of the fifteen topics predicts a number.

---
---
# 🎬 Part 1 — The Demo · blocks A to C

**Watch, do not type along.** Each block header says which subsections are walked live (🎤)
and which are reference (📖).

---
# A · From a Label to a Number
🎤 **Walked live: all of A.** It is the shortest path from what you already know to what changes.

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
print('ready')

### 🔵 A.1 — The same Pipeline, one step different

California housing: **20,640 districts**, and the target is the median house value in that district,
in hundreds of thousands of dollars.

Nothing about the structure changes from session 1. Impute, scale, model. **Only the last step is new** —
a `DecisionTreeRegressor` where a `DecisionTreeClassifier` used to be.

#### What the three steps are, since we build one every session

A **`Pipeline`** is a list of steps that run in order. Everything before the last one *transforms* the
data; the last one is the *model*. You call `.fit()` once, on the whole thing.

| Step | What it does | What it is protecting you from |
|---|---|---|
| `impute` | fills in missing values — here with the **median of that column** | a model that refuses to run, or silently drops rows |
| `scale` | rewrites every column to mean 0, spread 1, so **no column is "bigger" than another just because of its units** | a model that thinks population matters more than income because the numbers are larger |
| `model` | the part that actually learns | — |

**The reason it is a `Pipeline` and not three separate lines is leakage.** Each step learns its numbers
(the median to fill with, the mean and spread to scale by) **from the training data only**, and then
applies those same numbers to the test data. Do it by hand on the full table before splitting and the
test set has already leaked into your preprocessing — the scores come out better than the truth. That
was C.6 in session 1, and it is why the split happens above and the `Pipeline` below.

#### What `scale` computes, and why it matters at all

`StandardScaler` replaces every value with **how many standard deviations it sits from the mean of its
own column**:

```
        x - mean(column)
    z = ----------------
          std(column)
```

Both `mean` and `std` come from the **training set only** — that is the leakage rule above, applied.
One real value from this data, using the training mean 1426.4530 and training std 1137.0219:

```
    District A, Population = 2300
    z = (2300 - 1426.4530) / 1137.0219 = 0.7683      -> 0.77 std above average
```

Afterwards every column has mean 0 and standard deviation 1. Nothing about the *shape* of a column
changes — the order of the values, and who is far from whom, is untouched. Only the units go.

**Why any of that matters: because distance adds columns together.** A model like kNN measures how far
apart two districts are by taking the difference in every column, squaring each one, and summing. So a
column contributes in proportion to its *squared* size — and these columns are not remotely comparable:

| Column | one standard deviation is… | squared, that contributes |
|---|---:|---:|
| `Population` | 1137.0219 people | **1,292,819** |
| `AveBedrms` | 0.4332 rooms | **0.188** |

**An equally typical move in each column contributes about 6.9 million times more from `Population`
than from `AveBedrms`.** Unscaled, the distance between two districts is decided by population alone
and every other column is invisible — not because population matters more, but because it is counted
in people and bedrooms are counted in rooms. **After scaling, one standard deviation is worth exactly
1 in every column**, so each gets a say and the model decides which actually helps.

**A tree escapes all of this because it never adds columns together.** It looks at one column at a time
and asks *is this value above that threshold* — an ordering question, and scaling preserves order. That
is why the same scaler is worth `0` to the tree and worth half its error to kNN.

> **Neither transformer does anything visible on *this* dataset, and that is worth knowing rather than
> hiding.** California housing has **zero missing values**, so `impute` passes the data straight
> through. And a decision tree splits on *order*, never on distance, so scaling cannot move it —
> the MAE is `0.4482` with the scaler and `0.4482` without, to four decimals.
>
> **Both steps stay anyway, for two reasons.** Your own topic almost certainly has missing values, and
> having the slot already there means you do not rebuild the pipeline to add it. And the moment the
> model changes to anything that measures distance, `scale` stops being decoration: swap the tree for
> `KNeighborsRegressor` on this same data and the MAE goes from **0.4462 with the scaler to 0.8128
> without it** — nearly twice the error, from one line. *(Session 3 is where that gets its own block.)*

`pipe()` is just a shortcut so the next three blocks can say `pipe(SomeModel())` instead of retyping
the same three steps each time.

In [ ]:
h = fetch_california_housing(as_frame=True)
X, y = h.data, h.target

print('shape       :', h.frame.shape)
print('target      : min %.4f   median %.4f   max %.4f' % (y.min(), y.median(), y.max()))
print()
print(h.frame.head(3).round(3).to_string())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print()
print('train %d rows · test %d rows' % (len(X_train), len(X_test)))

✅ **Expected:** `(20640, 9)` · target runs from `0.1500` to `5.0000` with a median of `1.7970` ·
16,512 training rows and 4,128 test rows

**Hold on to that maximum of 5.0000.** It comes back in B.2 and it is not a coincidence.

In [ ]:
def pipe(model):
    return Pipeline([('impute', SimpleImputer(strategy='median')),
                     ('scale',  StandardScaler()),
                     ('model',  model)])

tree = pipe(DecisionTreeRegressor(max_depth=8, random_state=SEED)).fit(X_train, y_train)
pred = tree.predict(X_test)

print('first five predictions :', pred[:5].round(3))
print('first five actual      :', y_test.values[:5].round(3))

✅ **Expected:** two rows of numbers that are close but never equal.

**That is the whole difference.** A classifier is right or wrong; a regressor is only ever
*off by some amount*, and the rest of this session is about how to measure that amount honestly.

### 🔵 A.2 — The baseline changes too — and its R² goes negative

Session 1's baseline guessed the most common class. **There is no most common house price**, so the
dumbest possible answer becomes: *predict the training median, every time, for everyone.*

`DummyRegressor` does exactly that. Session 1 used `DummyClassifier`; this is the same idea with
the only strategy that makes sense for a number.

In [ ]:
dumb = DummyRegressor(strategy='median').fit(X_train, y_train)
base_pred = dumb.predict(X_test)

print('baseline = always predict the training median (%.4f)' % y_train.median())
print('  MAE  %.4f' % mean_absolute_error(y_test, base_pred))
print('  RMSE %.4f' % mean_squared_error(y_test, base_pred) ** 0.5)
print('  R2   %.4f' % r2_score(y_test, base_pred))

✅ **Expected:** median `1.7985` · MAE `0.8740` · RMSE `1.1731` · **R² `-0.0502`**

**Every year somebody asks why R² is negative, so here it is.**

R² measures how much of the variation your model explains, against a reference. **R² = 0 is not "no
model" — it is predicting the mean of the *test* set**, which you never get to see. Our baseline
predicts the *median of the training set*, which is a slightly different number on slightly different
data, so it lands a little worse than that reference and the score drops just below zero.

**A baseline R² near zero, on either side, is correct.** What would be alarming is a baseline R² of 0.4.

### 🔵 A.3 — Three metrics, three different questions

> **One of these three you have met, two you have not.** L04 uses **R²** throughout, and its residual
> analysis is what B.1 builds on. **MAE and RMSE are not in any lecture deck** — treat them as new.

Run the same three numbers on the baseline and on the tree, side by side.

In [ ]:
def report(name, y_pred):
    return {'model': name,
            'MAE':  mean_absolute_error(y_test, y_pred),
            'RMSE': mean_squared_error(y_test, y_pred) ** 0.5,
            'R2':   r2_score(y_test, y_pred)}

rows = [report('Baseline (train median)', pipe(DummyRegressor(strategy='median')).fit(X_train, y_train).predict(X_test)),
        report('Decision Tree (depth 8)', pred),
        report('Random Forest (200)',     pipe(RandomForestRegressor(n_estimators=200, random_state=SEED,
                                                                     n_jobs=-1)).fit(X_train, y_train).predict(X_test))]

print(pd.DataFrame(rows).round(4).to_string(index=False))

✅ **Expected**

| model | MAE | RMSE | R² |
|---|--:|--:|--:|
| Baseline (train median) | 0.8740 | 1.1731 | −0.0502 |
| Decision Tree (depth 8) | 0.4482 | 0.6497 | 0.6779 |
| Random Forest (200) | 0.3265 | 0.5038 | 0.8063 |

**What each column is actually telling you:**

- **MAE — the average size of a miss, in the units of the target.** 0.4482 means the tree is typically
  off by about \\$45,000. It is the only one of the three you can say out loud to somebody who does
  not do this for a living.
- **RMSE — the same idea, but it squares the errors first.** One large miss moves it far more than
  several small ones. Notice RMSE is always the larger number, and **the gap between MAE and RMSE tells
  you how lopsided your errors are**: 0.4482 against 0.6497 says a few big misses are doing the damage.
- **R² — the share of the variation explained**, against predicting the test mean. Unitless, so it is
  the one to compare across different datasets, and the one that means nothing on its own.

**Report your result against the baseline, not on its own:** the tree removes **48.7%** of the error the
do-nothing model makes. That sentence survives a question; `MAE 0.4482` does not.

### 🔵 A.4 — Choosing between them, from what an error costs

**The metric is a decision about consequences, not about which number looks better.** Ask one question:

> **Is one large error worse than several small ones that add up the same?**

| Your answer | The metric | Because |
|---|---|---|
| **Yes** — one district valued \\$200k out is worse than four out by \\$50k | **RMSE** | squaring makes the large miss dominate |
| **No** — the total is what matters | **MAE** | every error counts once, in real units |
| *"I need to compare against a different dataset"* | **R²** | unitless, but say what the baseline was |

**For house valuation a single large error usually is worse** — one wildly mispriced district is a real
problem, four slightly-off ones are noise. So RMSE is defensible here. **We report MAE alongside it**
because it is the number you can explain.

> **Decide before you train.** A metric chosen after you have seen the scores is a metric chosen to
> flatter them, and that is the difference the marking looks for.

---
# B · Reading the Errors
🎤 **Walked live:** B.1 · B.2 · B.3 &nbsp;·&nbsp; 📖 **read on your own:** B.4

Lecture 04 covers residual analysis on paper. This block is where you watch it happen on real data,
and where **the residual plot tells you something the three metrics cannot.**

### 🔵 B.1 — The residual plot: a cloud, or a shape

A **residual** is one number: actual minus predicted. Plot residuals against what the model predicted.

- **A shapeless cloud centred on zero** means the model's errors are random. That is as good as it gets.
- **Any shape at all** — a slope, a curve, a wall — means the model is wrong **in a pattern**, and a
  pattern is something you can go and fix.

In [ ]:
resid = y_test - pred

plt.figure(figsize=(7, 4.2))
plt.scatter(pred, resid, s=6, alpha=0.25, color='tab:blue')
plt.axhline(0, color='tab:red', lw=1)
plt.xlabel('predicted value'); plt.ylabel('residual  (actual - predicted)')
plt.title('Residuals fan out, and there is a hard diagonal edge on the right')
plt.tight_layout(); plt.show()

print('mean residual overall: %+.4f' % resid.mean())

✅ **Expected:** a mean residual of about `+0.0025` — near zero, as it should be — **and a plot that is
clearly not a shapeless cloud.**

Two things are visible:

1. **The spread widens as predictions get larger.** Cheap districts are predicted tightly; expensive
   ones are not. The model is less certain at the top and the single MAE figure hides that completely.
2. **A hard diagonal edge along the upper right.** Nothing in the metrics hinted at it. B.2 is what it is.

### 💥 B.2 — The target hit a ceiling, and the residuals show it

Look again at A.1: the maximum target value was **5.0000**. Not 4.97, not 5.13. Exactly the maximum.

**That is the signature of a censored variable** — the survey stopped recording above a cap and wrote
everything higher down as the cap. Count them.

In [ ]:
cap = y.max()
n_cap = (y == cap).sum()
print('the maximum value is exactly %r' % cap)
print('rows sitting on it          : %d of %d  (%.2f%%)' % (n_cap, len(y), n_cap / len(y) * 100))
print()
print(y.value_counts().sort_index(ascending=False).head(4).to_string())

capped = y_test == cap
print()
print('mean residual on capped rows : %+.4f' % resid[capped].mean())
print('mean residual on all the rest : %+.4f' % resid[~capped].mean())

✅ **Expected:** the maximum is `5.00001` · **965 rows (4.68%) sit exactly on it** · and the residuals split:

| rows | mean residual |
|---|--:|
| on the cap | **+0.8836** |
| everything else | −0.0374 |

**The model under-predicts the capped districts by nearly a whole unit, systematically** — and by a
rounding error everywhere else. That is the
diagonal edge in B.1.

**And it cannot be fixed by tuning.** A district worth \\$800k was written down as \\$500k before you
ever saw the file. No model recovers information the data does not contain — **the honest move is to say
so in your report**, and possibly to exclude the capped rows and say why.

> **This is what a residual plot is for.** Three metrics said the tree was decent. The residual plot
> said *4.7% of your data has had its answer erased*, which is a completely different conversation.

### 🔵 B.3 — Outliers: IQR, the fence, and why the formula is not the decision

Two terms first, because the lecture course never defines them.

#### IQR — where the middle half of the data lives

Sort a column and cut it into four equal parts. **Q1** is the value at 25%, **Q3** the value at 75%,
and **IQR = Q3 − Q1** is the range the middle 50% occupies.

**Why not the standard deviation?** Because one extreme value drags it and leaves the IQR untouched —
Q1 and Q3 do not care how large the largest value is, only how many values sit above and below them.

#### Fence — the line drawn 1.5 IQRs out from the box

```
lower fence = Q1 - 1.5 x IQR
upper fence = Q3 + 1.5 x IQR
```

Anything outside is **called an outlier by convention**. The 1.5 is Tukey's choice, not a theorem —
on bell-shaped data it flags about 0.7% of rows, which is a useful number of things to go and look at.
Use 3.0 and you flag only the extreme. **Neither is right; you choose and you say why.**

> **A box plot is exactly this drawn out.** The box spans Q1 to Q3, the line inside is the median, the
> whiskers reach the last point *inside* the fences, and everything past them is a flagged outlier.

In [ ]:
col = X['AveRooms']
q1, q3 = col.quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
flagged = col > upper

print('Q1 %.4f   Q3 %.4f   IQR %.4f' % (q1, q3, iqr))
print('upper fence %.4f' % upper)
print('flagged     %d of %d rows (%.2f%%) - the largest is %.2f' % (flagged.sum(), len(col),
                                                                    flagged.mean() * 100, col.max()))
print()
print('the three largest, with their district size:')
print(X.loc[col.sort_values(ascending=False).head(3).index,
            ['AveRooms', 'Population', 'AveOccup']].round(2).to_string())

plt.figure(figsize=(7, 2.4))
plt.boxplot(col, vert=False, widths=0.6)
plt.axvline(upper, color='tab:red', ls='--', lw=1)
plt.text(upper + 3, 1.28, 'IQR fence = %.1f' % upper, color='tab:red', fontsize=9)
plt.xlabel('average rooms per household')
plt.title('466 districts sit past the fence - and almost none of them are errors')
plt.yticks([]); plt.tight_layout(); plt.show()

✅ **Expected:** Q1 `4.4407` · Q3 `6.0524` · IQR `1.6117` · fence `8.4699` ·
**466 rows flagged (2.26%)**, and the largest district averages **141.91 rooms per household**

**Now look at the three largest.** Their populations are **30, 36 and 83 people.** These are tiny
districts where "average rooms per household" is computed from a handful of homes, so one large
property moves it enormously. **The value is not a typo — it is a real average of a very small group.**

**Outliers come in two kinds and the formula cannot tell them apart:**

| | What to do |
|---|---|
| **Recording errors** — an age of 999, a BMI of 0 | fix them, or drop them, and say which |
| **Rare but real values** — a 30-person district, a genuine mansion | **keep them**, and say why they are there |

Deleting everything past the fence here would remove 466 real districts and teach the model that
America has no small rural areas. **The formula finds candidates. You make the decision.**

### 📖 B.4 — Multicollinearity: two columns saying the same thing

Lecture 04 introduces this through the correlation matrix. Here is what it looks like on our data.

In [ ]:
corr = X.corr()
pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(corr.columns) for b in corr.columns[i + 1:]]
pairs.sort(key=lambda t: -abs(t[2]))

print('most correlated feature pairs:')
for a, b, v in pairs[:3]:
    print('  %-12s %-12s %+.4f' % (a, b, v))

print()
print('correlation of each feature with the target:')
print(X.corrwith(y).sort_values(key=abs, ascending=False).round(4).to_string())

✅ **Expected:** `Latitude` and `Longitude` at **−0.9247**, `AveRooms` and `AveBedrms` at **+0.8476**

**Latitude and longitude are almost perfectly anti-correlated here** because California runs on a
diagonal — go north and you also go west. The two columns are largely one piece of information.

**What it costs you:** with a tree, very little. With a linear model, the coefficients become unstable
and unreadable — swap which of the two is listed first and they can flip sign — so **any statement of
the form "latitude matters more than longitude" is not supported by this data.**

**And notice `MedInc` at 0.6881 against the target**, four times anything else. Whatever your model
does, median income is most of what it has to work with.

---
# C · Chart Polish, Step 2
📖 **Reference · the last ten minutes of the session**

Session 1's rung: **a chart title should be the conclusion**, and a y-axis that does not start at zero
magnifies small differences. Those still apply.

**This session's rung is the residual plot**, because it is the chart you will put on slide 9 and it is
the easiest one in this course to draw badly.

| Do | Why |
|---|---|
| **Equal limits above and below zero** | an unequal axis makes symmetric errors look one-sided |
| **A visible zero line** | it is the only reference the reader has |
| **Label the points that escape** | *"these are the capped districts"* beats an unexplained smear |
| **Fade the bulk, highlight what you are pointing at** | grey for the cloud, colour for the story |

In [ ]:
lim = np.abs(resid).max() * 1.05

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

# left: the default, drawn without thinking
ax[0].scatter(pred, resid, s=6)
ax[0].set_title('Residuals')                       # names the chart, says nothing
ax[0].set_xlabel('pred'); ax[0].set_ylabel('resid')

# right: the same numbers, drawn to make the point
ax[1].scatter(pred[~capped.values], resid[~capped.values], s=6, alpha=0.25, color='0.6', label='normal districts')
ax[1].scatter(pred[capped.values],  resid[capped.values],  s=10, alpha=0.7,  color='tab:red', label='at the 5.0 cap')
ax[1].axhline(0, color='black', lw=1)
ax[1].set_ylim(-lim, lim)
ax[1].set_xlabel('predicted median value'); ax[1].set_ylabel('actual - predicted')
ax[1].set_title('The 965 capped districts are under-predicted by ~0.9, every time')
ax[1].legend(frameon=False, fontsize=8)

plt.tight_layout(); plt.show()

✅ **Expected:** two panels from **the same residuals**, telling different stories.

The left one is what you get for free: a smear labelled "Residuals". The right one has equal limits so
the asymmetry is honest, a zero line to read against, the bulk faded to grey, **and the finding in the
title instead of the chart's name.**

**The right panel is a slide. The left panel is a cell output.** Requirement 5 marks the difference.

---
---
# 📋 Part 2 — Pick Your Topic
### Assignment 2 · one of these 15 · this takes ~8 minutes

**Groups of three or four. One topic per group, first come first served, no two groups on the same one.**

**Every topic here predicts a number.** Take yours and go the whole way:

```
your data -> EDA -> preprocessing (Pipeline) -> baseline() -> model -> a metric you can justify -> limitations
```

📄 **[How it is marked, and what to hand in](https://classes.incortx.com/DataAnalytics/session-00/)** — the short version: you need **a baseline beside
every number**, a **`Pipeline` with no leakage**, and **a metric chosen from what an error costs**.
A model that loses to its own baseline still scores full marks if you can explain why.

**The 15 topics.** 🟢 straightforward · 🟡 has something awkward in it

The **trap** column is not a general warning. It is the specific thing that will bite you in that
dataset, and it is where the questions will come from.

| # | Topic | Data | The trap | |
|:--:|---|---|---|:--:|
| **1** | What is this district of houses worth? | `fetch_california_housing()` | **The target is capped at 5.0** -- 965 of 20,640 districts (4.7%) sit exactly on the ceiling, and no amount of tuning will ever get those right | 🟢 |
| **2** | What should this diamond cost? | `sns.load_dataset('diamonds')` | The `cut`/`color`/`clarity` grades are ordinal and **sort into the wrong order alphabetically** -- `IF`, the best clarity there is, lands second | 🟢 |
| **3** | How much forest will this fire burn? | `archive.ics.uci.edu/static/public/162/forest+fires.zip` | **247 of 517 rows have target = 0.00.** Predicting zero every time looks respectable on average -- you have to decide what you are actually predicting | 🟡 |
| **4** | How many bikes will be hired this hour? | `archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip` (use `hour.csv`) | `casual` + `registered` equals the target **exactly, on all 17,379 rows.** Leave either one in and you score almost perfectly without predicting anything | 🟢 |
| **5** | How much fuel does this car use? | `sns.load_dataset('mpg')` | `name` is unique on 305 of 398 rows -- feeding it in lets the model memorise cars one at a time. And `horsepower` has 6 NaNs you have to decide about | 🟢 |
| **6** | Predict diabetes progression one year ahead | `sklearn.datasets.load_diabetes()` | The features **arrive already standardised** (every column has mean zero), so the coefficients you get cannot be read back in real units | 🟡 |
| **7** | How much load will this concrete mix take? | `archive.ics.uci.edu/static/public/165/concrete+compressive+strength.zip` | The ingredients are **compositional** -- they sum to a fixed weight, so the features are not independent. And curing age acts logarithmically, not linearly | 🟡 |
| **8** | How much heating energy does this building need? | `archive.ics.uci.edu/static/public/242/energy+efficiency.zip` | **There are two targets** (heating load and cooling load). Pick one or go multi-output -- pick the wrong one and your numbers cannot be compared with anyone else's | 🟢 |
| **9** | What is this Taipei apartment worth? | `archive.ics.uci.edu/static/public/477/real+estate+valuation+data+set.zip` | Only 414 rows, and **`transaction date` is a decimal year** (2013.250). Use it raw and the model learns the time trend instead of the location | 🟢 |
| **10** | How old is this abalone? | `archive.ics.uci.edu/static/public/1/abalone.zip` | **It looks like classification and is really ordinal** -- being wrong by 1 year and being wrong by 10 should not cost the same | 🟡 |
| **11** | How loud is this wing shape? | `archive.ics.uci.edu/static/public/291/airfoil+self+noise.zip` | Unusually clean, not a single missing value -- **which makes it the topic with no excuses.** A poor result means the model or the metric is wrong, not the data | 🟢 |
| **12** | How many bikes will Seoul hire? | `archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip` | Encoded as **cp949, not utf-8** -- opening it directly fails. Fix with `pd.read_csv(f, encoding="cp949")`. Holidays and seasons also have to be turned into features yourself | 🟡 |
| **13** | What grade will this student get? (as a number) | `archive.ics.uci.edu/static/public/320/student+performance.zip` | **G1 and G2 predict G3 almost perfectly (corr 0.905)** -- keeping them gives a lovely score and tells you nothing. Decide, and explain the decision | 🟢 |
| **14** | What score will this wine get? (as regression) | `archive.ics.uci.edu/static/public/186/wine+quality.zip` | The target is an integer from 3 to 8. **Treat it as regression and you get decimals -- decide whether to round**, and know that rounding moves the score a lot | 🟡 |
| **15** | What should this house sell for? | `fetch_openml('house_prices', version=1)` (Ames, 80 columns) | **80 columns, several of them more than half empty** -- you decide column by column what to keep. The heaviest preprocessing job in the bank | 🟡 |

In [ ]:
# ── Record your group's choice ─────────────────────────────────────────────
TOPIC_ID = None      # <- put your group's topic number here, then run this cell

TOPIC_TRAPS = {
     1: ('What is this district of houses worth?',
        '**The target is capped at 5.0** -- 965 of 20,640 districts (4.7%) sit exactly on the ceiling, and no amount of tuning will ever get those right'),
     2: ('What should this diamond cost?',
        'The `cut`/`color`/`clarity` grades are ordinal and **sort into the wrong order alphabetically** -- `IF`, the best clarity there is, lands second'),
     3: ('How much forest will this fire burn?',
        '**247 of 517 rows have target = 0.00.** Predicting zero every time looks respectable on average -- you have to decide what you are actually predicting'),
     4: ('How many bikes will be hired this hour?',
        '`casual` + `registered` equals the target **exactly, on all 17,379 rows.** Leave either one in and you score almost perfectly without predicting anything'),
     5: ('How much fuel does this car use?',
        '`name` is unique on 305 of 398 rows -- feeding it in lets the model memorise cars one at a time. And `horsepower` has 6 NaNs you have to decide about'),
     6: ('Predict diabetes progression one year ahead',
        'The features **arrive already standardised** (every column has mean zero), so the coefficients you get cannot be read back in real units'),
     7: ('How much load will this concrete mix take?',
        'The ingredients are **compositional** -- they sum to a fixed weight, so the features are not independent. And curing age acts logarithmically, not linearly'),
     8: ('How much heating energy does this building need?',
        "**There are two targets** (heating load and cooling load). Pick one or go multi-output -- pick the wrong one and your numbers cannot be compared with anyone else's"),
     9: ('What is this Taipei apartment worth?',
        'Only 414 rows, and **`transaction date` is a decimal year** (2013.250). Use it raw and the model learns the time trend instead of the location'),
    10: ('How old is this abalone?',
        '**It looks like classification and is really ordinal** -- being wrong by 1 year and being wrong by 10 should not cost the same'),
    11: ('How loud is this wing shape?',
        'Unusually clean, not a single missing value -- **which makes it the topic with no excuses.** A poor result means the model or the metric is wrong, not the data'),
    12: ('How many bikes will Seoul hire?',
        'Encoded as **cp949, not utf-8** -- opening it directly fails. Fix with `pd.read_csv(f, encoding="cp949")`. Holidays and seasons also have to be turned into features yourself'),
    13: ('What grade will this student get? (as a number)',
        '**G1 and G2 predict G3 almost perfectly (corr 0.905)** -- keeping them gives a lovely score and tells you nothing. Decide, and explain the decision'),
    14: ('What score will this wine get? (as regression)',
        'The target is an integer from 3 to 8. **Treat it as regression and you get decimals -- decide whether to round**, and know that rounding moves the score a lot'),
    15: ('What should this house sell for?',
        '**80 columns, several of them more than half empty** -- you decide column by column what to keep. The heaviest preprocessing job in the bank'),
}

if TOPIC_ID in TOPIC_TRAPS:
    title, trap = TOPIC_TRAPS[TOPIC_ID]
    print('Topic %d: %s' % (TOPIC_ID, title))
    print('Watch out for : %s' % trap)
else:
    print('Set TOPIC_ID to your group number (1-15) and run this cell again.')

✅ **Expected:** your topic and its trap printed back at you. Write the trap somewhere you will see it
again — it is the first thing to check when your results look strange.

---
---
# 🟠 Part 3 — Your Hour · 60 Minutes, Your Own Data

**Everything above was the demo.** From here it is your group's work, and it is what gets marked.

This section does not depend on a single cell above it. Run it from the top of this section and it works.

### The steps, and the clock

| Minutes | Step | What has to exist when you are done |
|:--:|---|---|
| — | **0 · Data already loaded** | you did this before class. `shape` · `head()` · one sentence on what a row is |
| 0–12 | **1 · EDA → one insight** | a chart, and a sentence stating what you *found* |
| 12–27 | **2 · Prepare the data** | what was wrong, what you did, **and why that choice** — inside a `Pipeline` |
| 27–37 | **3 · Metric + baseline** | the metric **with a reason**, and `baseline()` measured the same way |
| 37–50 | **4 · Today's technique** *(if you get there)* | a score, printed next to the baseline |
| 50–60 | **5 · Write up + get ready** | one sentence, one limitation, notebook scrolled to where you start |

**Steps 1, 2 and 3 are what you present and what is marked. Step 4 is a bonus.**

**Because loading happened before class, this hour is less rushed than session 1's.** Use the slack on
step 1 — a residual plot you actually read beats a model you cannot explain.

In [ ]:
# ── SUBMISSION HEADER — fill this in first ─────────────────────────────────
GROUP     = ''            # your group letter: 'A' .. 'J'
MEMBERS   = ['', '', '']  # everyone in the group - keep this order all term
TOPIC_ID  = None          # the topic number your group claimed

IN_CLASS  = ['', '']      # the TWO who present in the room, start of next session
IN_CLIP   = ['']          # the rest, presenting in the homework video

# ── check: everyone presents somewhere ─────────────────────────────────────
_all   = [m.strip() for m in MEMBERS  if m.strip()]
_room  = [m.strip() for m in IN_CLASS if m.strip()]
_clip  = [m.strip() for m in IN_CLIP  if m.strip()]

print(f'Group {GROUP or "?"} | topic {TOPIC_ID} | {len(_all)} members')
print(f'  in the room : {", ".join(_room) or "-- nobody --"}')
print(f'  in the video: {", ".join(_clip) or "-- nobody --"}')

missing = [m for m in _all if m not in _room + _clip]
twice   = [m for m in _room if m in _clip]
if not GROUP or not _all or TOPIC_ID is None:
    print('\n[ ] header not filled in yet')
elif missing:
    print(f'\n[!] not presenting anywhere: {", ".join(missing)} - everyone has to present')
elif twice:
    print(f'\n[!] listed twice: {", ".join(twice)} - pick one or the other')
elif len(_room) != 2:
    print(f'\n[!] {len(_room)} in the room, should be 2')
else:
    print('\n[ok] everyone presents')

### 🟠 Setup for this section

Its own imports and its own helpers, so this half runs whatever happened above.

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import io as _io, zipfile, urllib.request      # several topics are zips this session
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing, fetch_openml, load_diabetes
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
pd.set_option('display.width', 120); pd.set_option('display.max_columns', 30)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
print('ready')

**Two helpers come with the notebook: `eda()` and `baseline()`.** Run the cell below once.

**Neither does the thinking.** `eda()` prints the six views you should always look at; `baseline()`
prints the number your model has to beat. **Reading them is the marked part.**

In [ ]:
def eda(df, target=None, n=6):
    """Print the six things you should always look at first. Reading them is your job.

        eda(df)                    # no target column yet
        eda(df, target='outcome')  # adds class balance + correlation with the target
    """
    import pandas as _pd
    line = '\u2500' * 62

    print(line); print(f'1. SHAPE      {df.shape[0]:,} rows x {df.shape[1]} columns')
    print(line); print('2. ONE ROW    what does a single row actually represent?')
    print(df.head(3).to_string())

    print(line); print('3. TYPES      a number stored as text will not go into a model')
    info = _pd.DataFrame({'dtype': df.dtypes.astype(str),
                          'non_null': df.notna().sum(),
                          'distinct': df.nunique()})
    print(info.to_string())

    print(line); print('4. MISSING    how much, and in which columns')
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    if len(miss) == 0:
        print('   isna() finds none  --  but a 0 or a -200 can be a missing value in disguise.')
        print('   Check step 5 for impossible values before you believe this.')
    else:
        print(_pd.DataFrame({'missing': miss, 'percent': (miss / len(df) * 100).round(1)}).to_string())

    print(line); print('5. RANGES     look for a min or max that cannot be real')
    num = df.select_dtypes('number')
    print(num.describe().T[['min', '25%', '50%', '75%', 'max']].to_string() if len(num.columns)
          else '   no numeric columns')

    if target is not None and target in df.columns:
        print(line); print(f'6. TARGET     {target!r}')
        y = df[target]
        if y.dtype.kind == 'f' or y.nunique() > 20:
            print(y.describe().to_string())
            corr = num.corr(numeric_only=True)[target].drop(target).sort_values(key=abs, ascending=False)
            print(f'\n   strongest correlations with {target}:')
            print(corr.head(n).round(4).to_string())
        else:
            print(y.value_counts(normalize=True).round(4).to_string())
            print(f'\n   most common class is {y.value_counts(normalize=True).max():.1%} of rows')
    print(line)
    print('Now write ONE sentence about something you did not know 60 seconds ago.')

def baseline(y_train, y_test, kind='auto'):
    """Print the score of the dumbest possible model for this target.

    You do not have to write a baseline. You do have to read it and say what it
    means -- that is the part that carries marks.

        baseline(y_tr, y_te)            # works out classification vs regression
        baseline(y_tr, y_te, 'clf')     # force it
        baseline(y_tr, y_te, 'reg')
    """
    import numpy as _np, pandas as _pd
    from sklearn.dummy import DummyClassifier, DummyRegressor
    from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error,
                                 mean_squared_error, r2_score)

    ytr, yte = _pd.Series(y_train), _pd.Series(y_test)
    if kind == 'auto':
        numeric = _pd.api.types.is_numeric_dtype(ytr)
        kind = 'reg' if (numeric and (ytr.dtype.kind == 'f' or ytr.nunique() > 20)) else 'clf'
        looks = 'a number -> regression' if kind == 'reg' else 'a label -> classification'
        print(f'[auto] your target looks like {looks}'
              f'  ({ytr.nunique()} distinct values, dtype {ytr.dtype})')
        print("       wrong guess? pass kind='clf' or kind='reg'")

    Xtr = _np.zeros((len(ytr), 1))          # a baseline ignores the features on purpose
    Xte = _np.zeros((len(yte), 1))

    if kind == 'clf':
        m = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr)
        p = m.predict(Xte)
        top = m.classes_[0]
        top = top.item() if hasattr(top, 'item') else top
        acc = accuracy_score(yte, p)
        f1 = f1_score(yte, p, average='binary' if yte.nunique() == 2 else 'macro',
                      zero_division=0)
        print(f'baseline = always predict the most common class ({top!r})')
        print(f'  accuracy {acc:.4f}')
        print(f'  F1       {f1:.4f}   <- same model. If these two disagree, accuracy is the wrong metric')
        return {'accuracy': acc, 'f1': f1}

    m = DummyRegressor(strategy='median').fit(Xtr, ytr)
    p = m.predict(Xte)
    mae = mean_absolute_error(yte, p)
    rmse = mean_squared_error(yte, p) ** 0.5
    r2 = r2_score(yte, p)
    print(f'baseline = always predict the training median ({_np.median(ytr):.4f})')
    print(f'  MAE  {mae:.4f}')
    print(f'  RMSE {rmse:.4f}')
    print(f'  R2   {r2:.4f}   <- a baseline R2 at or just below zero is correct, not a bug')
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

print('eda() and baseline() ready')

### 🟠 Opening a zip, if your topic is one

**Eleven of the fifteen topics are zips this session.** You should have loaded yours before class, but
here is the shape in one place.

```python
with urllib.request.urlopen(URL) as response:
    z = zipfile.ZipFile(_io.BytesIO(response.read()))
print(z.namelist())                         # always look before you read
df = pd.read_csv(z.open('the_file.csv'), sep=';')

# a zip inside a zip
inner = zipfile.ZipFile(_io.BytesIO(z.read('inner.zip')))
df = pd.read_csv(inner.open('bank-full.csv'), sep=';')
```

**Topic 12 is encoded `cp949`, not utf-8** — `pd.read_csv(f, encoding='cp949')`.
Session 1's block B has the full four-shapes reference if you need it.

### 🟠 Step 0 — Your data, already loaded

Load it and run `eda()` straight away. It takes a fraction of a second and prints all six views, so you
begin step 1 already knowing where to look.

In [ ]:
# TODO: load your group's data into `df`, then run eda() on it

df = None
# eda(df, target='...')

✅ **What topic 5's `eda()` hands you in under a second**

- **view 3 shows `name` with 305 distinct values across 398 rows** — nearly unique. Feed that to a model
  and it memorises cars one at a time.
- **view 4 shows `horsepower` missing 6 values.** Small, but you have to decide something.
- **view 6 shows the target's spread**, and which features move with it.

**Write the one sentence:** what is one row of this data?

### 🟠 Step 1 — EDA → one insight · *0–12 min*

**The chart is not the deliverable. The sentence under it is.**

For a number-valued target, the two charts that earn their place are a **histogram of the target**
(is it skewed? does it stop dead at a ceiling, like B.2?) and a **scatter of the target against your
strongest feature** (is the relationship even a straight line?).

| Not an insight | An insight |
|---|---|
| "This is a histogram of the target." | "The target stops dead at 5.0 with 965 rows on it — those are censored and no model can recover them." |
| "mpg and weight are correlated." | "Heavier cars use more fuel, but the relationship bends — a straight line will over-predict at both ends." |

In [ ]:
# TODO: one chart that shows something about your target

**What I found:** *(one sentence — a finding, not a description of the chart)*

**What this changes about what I do next:** *(write it here)*

### 🟠 Step 2 — Prepare the data · *12–27 min*

Everything step 1 told you was wrong, you now fix — **and you write down why you fixed it that way.**

**Two hard rules, and breaking either one costs you:**

1. **Put it in a `Pipeline`, do not edit `df` in place.** A `Pipeline` learns its fill values and
   scaling from the training rows only, which is the whole point.
2. **`train_test_split` comes before any `fit`.**

> **Regression adds one decision session 1 did not have: what to do about the columns that leak.**
> If a column could only be known *after* the answer, it has to go — the same `alive` problem, wearing
> a different hat. Topic 4's `casual + registered` and topic 13's `G1`/`G2` are exactly this.

In [ ]:
# TODO: split X/y, drop what leaks, then build the Pipeline

**What I fixed, and why I chose that fix:** *(one line per decision)*

### 🟠 Step 3 — Metric + baseline · *27–37 min*

**Every topic this session predicts a number**, so the choice is MAE, RMSE, or both.

Ask A.4's question about **your** data: *is one large error worse than several small ones?*
Predicting a price, usually yes. Predicting a count that gets summed later, usually no.

**The baseline code came with the notebook.** One call:

```python
base = baseline(y_train, y_test)
```

**Two rules that still belong to you:**

1. **Call it with the same `y_train` / `y_test` you gave your model.**
2. **From here on, the baseline goes next to every number you print.**

**Then read what came back** — a baseline R² at or just below zero is correct, not a bug (A.2).

In [ ]:
# TODO: split, name your metric and why, then call baseline()

✅ **Expected on topic 5:** median `22.4500` · MAE `5.9788` · RMSE `7.3652` · **R² `-0.0089`**

**Our metric is ___ because ___** *(fill this in — it is half of what requirement 1 looks for)*

### 🟠 Step 4 — Today's technique · *37–50 min* — **if you get there**

**This step is a bonus, not a requirement.** Your `prep` from step 2 is already built — drop a regressor
on the end of it and score it **next to the baseline, never alone.**

In [ ]:
# TODO: put a model on the end of `prep`, and score it against the baseline

✅ **Expected on topic 5:**

| Model | MAE | RMSE | R² |
|---|--:|--:|--:|
| Baseline (median) | 5.9788 | 7.3652 | −0.0089 |
| Decision Tree (depth 4) | 2.3302 | 3.3721 | 0.7885 |
| Decision Tree (depth 8) | 2.1330 | 3.2953 | 0.7980 |
| Random Forest (200) | 1.5955 | 2.1912 | 0.9107 |

**Two rows of this table are your slide 8.** Depth 4 against depth 8 is a comparison of two settings —
deeper helps a little here, unlike titanic in session 1 where it hurt. **Say which way it moved and
why**, and that page is done.

### 🟠 Step 5 — Write it up and close the file properly · *50–60 min*

Two sentences, both short. **You are not presenting today**, so the rest of this goes into making the
file usable when you reopen it at home:

1. **Write down what you would try next**, while the reason is still in your head.
2. **Say where you got stuck and what you had already ruled out.** An unfinished step costs nothing;
   an unfinished step you cannot describe costs the write-up mark.

**What we found:** *(one sentence — what does your number actually mean for the question you asked?)*

**What we do not trust:** *(one limitation — something about the data or the method)*

**Where we got stuck:** *(if a step defeated you, say which and why — this is worth writing down)*

---
## 🟠 After the hour — presenting, and handing in

**Everything about how this is presented, questioned and marked lives in one place, and it is not this
notebook:**

📄 **[How the assignment works, and what to hand in](https://classes.incortx.com/DataAnalytics/session-00/)**

That page is the only version — it covers the four minutes on screen, the code questions, asking
questions while other groups present, and every requirement for the hand-in. **Read it once at the
start of term, and again before your first turn.**

The three dates you need, and nothing else:

| When | What |
|---|---|
| **Before you leave today** | this notebook, as a file — *File → Download → Download .ipynb* |
| **The day before session 3** | the homework: notebook, slides, video, README — **as files, no links** |
| **The start of session 3** | you present, from the notebook you handed in |

---
# Session 2 Wrap-Up

### Four things to remember

1. **A number-valued answer is never right, only close.** Everything this session was about measuring
   *how close*, honestly.
2. **MAE, RMSE and R² answer different questions.** Pick from what an error costs — before you train.
3. **The residual plot sees what the metrics cannot.** Three respectable numbers hid the fact that
   4.7% of the target had been erased before we got it (B.2).
4. **The IQR fence finds candidates, not errors.** 466 flagged districts, and almost all of them real.

### What this session's notebook should end up carrying
- a **metric with a reason**, and `baseline()` printed beside every score
- a **residual plot you can read out loud**
- a line saying where you got to, and what you would try next

*(Dates, file formats and everything else about handing in: see the section above.)*

### Next session — no answer key at all
Two sessions of showing the model the right answers. **Next time there are none** — you group the data
without being told what the groups are, and the hard part becomes knowing whether you found anything real.